# 03 — Visualisation

**Project:** Urban Heat Island Satellite Analysis — London, UK  
**Author:** Zari Syed  
**Date:** 2025

---

## Overview

This notebook produces all publication-quality figures for the project:

1. Side-by-side LST maps for 2015 and 2023 with borough boundaries
2. ΔLST change map (2023 − 2015) with borough overlay
3. NDVI comparison maps (2015 vs 2023)
4. Top 5 hotspot boroughs bar chart
5. NDVI vs LST scatter plots (2015 and 2023)
6. Borough-level ΔLST vs ΔNDVI scatter

All figures are exported to `outputs/figures/` as 300 DPI PNG files.

**Inputs:** Processed rasters in `data/processed/`, tables in `outputs/tables/`

## 1. Imports and Configuration

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show as rioshow
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib_scalebar.scalebar import ScaleBar   # pip install matplotlib-scalebar
import contextily as ctx
from scipy import stats

warnings.filterwarnings('ignore')

# ── Matplotlib defaults ────────────────────────────────────────────────────
matplotlib.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
    'figure.dpi'        : 150,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'savefig.pad_inches': 0.1,
})

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
PROCESSED    = PROJECT_ROOT / 'data'    / 'processed'
SHAPEFILES   = PROJECT_ROOT / 'shapefiles'
TABLES       = PROJECT_ROOT / 'outputs' / 'tables'
FIGURES      = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Figures will be saved to: {FIGURES}")

In [ ]:
# ── Helper: load raster as (array, extent, crs) ───────────────────────────
def load_raster(path: Path) -> tuple[np.ndarray, list, object]:
    """Return (data_array, [left,right,bottom,top], crs)."""
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
        data[data == src.nodata] = np.nan
        b = src.bounds
        extent = [b.left, b.right, b.bottom, b.top]
        crs = src.crs
    return data, extent, crs


# ── Load all derived rasters ──────────────────────────────────────────────
lst_2015,   ext, crs_bng = load_raster(PROCESSED / 'lst_2015_london.tif')
lst_2023,   _,  _        = load_raster(PROCESSED / 'lst_2023_london.tif')
ndvi_2015,  _,  _        = load_raster(PROCESSED / 'ndvi_2015_london.tif')
ndvi_2023,  _,  _        = load_raster(PROCESSED / 'ndvi_2023_london.tif')
delta_lst,  _,  _        = load_raster(PROCESSED / 'delta_lst_2023_2015.tif')
delta_ndvi, _,  _        = load_raster(PROCESSED / 'delta_ndvi_2023_2015.tif')

# ── Load boroughs ─────────────────────────────────────────────────────────
boroughs = gpd.read_file(SHAPEFILES / 'london_boroughs.shp').to_crs(epsg=27700)

# ── Load summary table ────────────────────────────────────────────────────
summary = pd.read_csv(TABLES / 'borough_lst_ndvi_summary.csv')

print("All data loaded.")

## 2. Side-by-Side LST Maps (2015 vs 2023)

Panel maps showing the spatial distribution of Land Surface Temperature in both years, with borough boundaries overlaid.

In [ ]:
# Shared colour scale: use combined min/max across both years
lst_vmin = np.nanpercentile(np.concatenate([lst_2015.ravel(), lst_2023.ravel()]), 2)
lst_vmax = np.nanpercentile(np.concatenate([lst_2015.ravel(), lst_2023.ravel()]), 98)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, data, year in zip(axes, [lst_2015, lst_2023], ['2015', '2023']):
    im = ax.imshow(
        data, cmap='inferno', vmin=lst_vmin, vmax=lst_vmax,
        extent=ext, origin='upper'
    )
    boroughs.boundary.plot(ax=ax, color='white', linewidth=0.6, alpha=0.7)
    ax.set_title(f'Land Surface Temperature — {year}', fontweight='bold')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.tick_params(labelsize=9)

# Shared colourbar
cbar = fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.03, pad=0.02)
cbar.set_label('LST (°C)', fontsize=11)

fig.suptitle(
    'Urban Heat Island — Land Surface Temperature\nGreater London, Landsat 8/9 (July/August)',
    fontsize=14, fontweight='bold', y=1.02
)

out = FIGURES / 'fig01_lst_comparison.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 3. ΔLST Change Map (2023 − 2015)

A diverging colourmap centred at zero highlights areas of warming (positive) and cooling (negative).

In [ ]:
# Diverging norm centred at 0
d_abs = np.nanpercentile(np.abs(delta_lst), 98)
div_norm = TwoSlopeNorm(vmin=-d_abs, vcenter=0, vmax=d_abs)

fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)

im = ax.imshow(
    delta_lst, cmap='RdBu_r', norm=div_norm,
    extent=ext, origin='upper'
)
boroughs.boundary.plot(ax=ax, color='black', linewidth=0.7, alpha=0.6)

# Annotate top 5 hotspot boroughs
top5 = summary.nlargest(5, 'delta_lst')
boroughs_top5 = boroughs[boroughs['NAME'].isin(top5['NAME'])]
for _, row in boroughs_top5.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row['NAME'], xy=(centroid.x, centroid.y),
        fontsize=8, ha='center', color='black',
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none')
    )

cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.01)
cbar.set_label('ΔLST (°C)', fontsize=11)

ax.set_title(
    'Change in Land Surface Temperature\n2023 − 2015, Greater London',
    fontweight='bold', fontsize=13
)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

out = FIGURES / 'fig02_delta_lst.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 4. ΔLST Map with Contextily Basemap

Adding a basemap tile (OpenStreetMap) provides geographic context. Note: this requires an internet connection at runtime. The raster and borough GDF must be in Web Mercator (EPSG:3857) for contextily.

In [ ]:
import contextily as ctx
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.crs import CRS

# Reproject boroughs to Web Mercator for contextily
boroughs_wm = boroughs.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(11, 10), constrained_layout=True)

# Plot the ΔLST with a semi-transparent alpha so basemap shows through
# We need delta_lst reprojected to EPSG:3857 — use geopandas raster workaround
# via earthpy or directly plot borough-level delta on map

# ── Merge ΔLST to borough GDF ─────────────────────────────────────────────
boroughs_delta = boroughs_wm.merge(summary[['NAME', 'delta_lst']], on='NAME')

boroughs_delta.plot(
    column    ='delta_lst',
    cmap      ='RdBu_r',
    norm      =TwoSlopeNorm(vmin=boroughs_delta.delta_lst.min(), vcenter=0,
                            vmax=boroughs_delta.delta_lst.max()),
    ax        =ax,
    alpha     =0.75,
    edgecolor ='black',
    linewidth =0.5,
    legend    =True,
    legend_kwds={
        'label'      : 'Mean ΔLST (°C)',
        'orientation': 'vertical',
        'shrink'     : 0.6,
    }
)

# Add borough name labels for top 5
for _, row in boroughs_delta[boroughs_delta['NAME'].isin(top5['NAME'])].iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NAME'], xy=(c.x, c.y), fontsize=8, ha='center',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec='none'))

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=10)

ax.set_title(
    'Mean Borough ΔLST (2023 − 2015) — Greater London\nTop 5 hotspot boroughs labelled',
    fontweight='bold'
)
ax.set_axis_off()

out = FIGURES / 'fig03_delta_lst_basemap.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 5. NDVI Comparison Maps (2015 vs 2023)

In [ ]:
ndvi_vmin = np.nanpercentile(np.concatenate([ndvi_2015.ravel(), ndvi_2023.ravel()]), 2)
ndvi_vmax = np.nanpercentile(np.concatenate([ndvi_2015.ravel(), ndvi_2023.ravel()]), 98)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)

for ax, data, year in zip(axes, [ndvi_2015, ndvi_2023], ['2015', '2023']):
    im = ax.imshow(
        data, cmap='RdYlGn', vmin=ndvi_vmin, vmax=ndvi_vmax,
        extent=ext, origin='upper'
    )
    boroughs.boundary.plot(ax=ax, color='black', linewidth=0.5, alpha=0.6)
    ax.set_title(f'NDVI — {year}', fontweight='bold')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.tick_params(labelsize=9)

cbar = fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.03, pad=0.02)
cbar.set_label('NDVI', fontsize=11)

fig.suptitle(
    'Normalised Difference Vegetation Index (NDVI)\nGreater London, 2015 vs 2023',
    fontsize=14, fontweight='bold', y=1.02
)

out = FIGURES / 'fig04_ndvi_comparison.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 6. Top 5 Hotspot Boroughs — Bar Chart

In [ ]:
top10 = summary.nlargest(10, 'delta_lst').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

colours = ['#d7191c' if i < 5 else '#fdae61' for i in range(len(top10))]

bars = ax.barh(
    top10['NAME'][::-1], top10['delta_lst'][::-1],
    color=colours[::-1], edgecolor='white', linewidth=0.5
)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Mean ΔLST (°C)', fontsize=11)
ax.set_title(
    'Top 10 London Boroughs by Mean LST Change (2023 − 2015)\n'
    'Red = top 5 hotspots',
    fontweight='bold'
)

# Value labels
for bar, val in zip(bars[::-1], top10['delta_lst']):
    ax.text(
        val + 0.02, bar.get_y() + bar.get_height() / 2,
        f'{val:+.2f}°C', va='center', fontsize=9
    )

ax.spines[['top', 'right']].set_visible(False)

out = FIGURES / 'fig05_hotspot_boroughs.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 7. LST vs NDVI Scatter Plots

Pixel-level scatter plots for 2015 and 2023 showing the inverse relationship between vegetation density and surface temperature. We subsample pixels for plotting performance.

In [ ]:
SAMPLE_N = 50_000   # Number of pixels to sample for scatter plot

def sample_valid_pixels(
    arr_x : np.ndarray,
    arr_y : np.ndarray,
    n     : int = SAMPLE_N,
    seed  : int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """Randomly sample n co-valid pixel pairs."""
    valid = ~np.isnan(arr_x) & ~np.isnan(arr_y)
    x = arr_x[valid].ravel()
    y = arr_y[valid].ravel()
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(x), size=min(n, len(x)), replace=False)
    return x[idx], y[idx]


fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, lst, ndvi, year in zip(
    axes,
    [lst_2015, lst_2023],
    [ndvi_2015, ndvi_2023],
    ['2015', '2023']
):
    x_ndvi, y_lst = sample_valid_pixels(ndvi, lst)

    # Hex bin density scatter
    hb = ax.hexbin(
        x_ndvi, y_lst, gridsize=60, cmap='YlOrRd',
        mincnt=1, linewidths=0.1
    )
    fig.colorbar(hb, ax=ax, label='Pixel count')

    # Regression line
    slope, intercept, r, p, _ = stats.linregress(x_ndvi, y_lst)
    x_fit = np.linspace(np.nanmin(x_ndvi), np.nanmax(x_ndvi), 200)
    ax.plot(x_fit, slope * x_fit + intercept, 'b-', linewidth=2, label=f'r = {r:.3f}')

    ax.set_xlabel('NDVI', fontsize=11)
    ax.set_ylabel('LST (°C)', fontsize=11)
    ax.set_title(f'LST vs NDVI — {year}', fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    'Pixel-level LST vs NDVI — Greater London\n(hexbin density, 50k random sample)',
    fontsize=13, fontweight='bold'
)

out = FIGURES / 'fig06_lst_ndvi_scatter.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

## 8. Borough-Level ΔLST vs ΔNDVI Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

sc = ax.scatter(
    summary['delta_ndvi'], summary['delta_lst'],
    c=summary['delta_lst'], cmap='RdBu_r',
    norm=TwoSlopeNorm(
        vmin=summary.delta_lst.min(), vcenter=0, vmax=summary.delta_lst.max()
    ),
    s=80, edgecolors='black', linewidths=0.5, zorder=3
)
fig.colorbar(sc, ax=ax, label='Mean ΔLST (°C)')

# Regression
slope, intercept, r, p, _ = stats.linregress(
    summary['delta_ndvi'].dropna(),
    summary['delta_lst'].dropna()
)
x_fit = np.linspace(summary.delta_ndvi.min(), summary.delta_ndvi.max(), 200)
ax.plot(x_fit, slope * x_fit + intercept, 'k--', linewidth=1.5,
        label=f'r = {r:.3f},  p = {p:.3f}')

# Label top 5 hotspots
for _, row in summary.nlargest(5, 'delta_lst').iterrows():
    ax.annotate(
        row['NAME'],
        xy=(row['delta_ndvi'], row['delta_lst']),
        xytext=(6, 3), textcoords='offset points',
        fontsize=8
    )

ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
ax.axvline(0, color='grey', linewidth=0.8, linestyle=':')
ax.set_xlabel('Mean ΔNDVI (2023 − 2015)', fontsize=11)
ax.set_ylabel('Mean ΔLST (°C, 2023 − 2015)', fontsize=11)
ax.set_title(
    'Borough-Level Change in LST vs NDVI (2015–2023)\nGreater London Boroughs',
    fontweight='bold'
)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

out = FIGURES / 'fig07_borough_delta_scatter.png'
fig.savefig(out)
print(f"Saved: {out.name}")
plt.show()

---

## Summary of Exported Figures

| Figure | File | Description |
|---|---|---|
| Fig 1 | `fig01_lst_comparison.png` | Side-by-side LST maps, 2015 & 2023 |
| Fig 2 | `fig02_delta_lst.png` | ΔLST raster map with borough overlay |
| Fig 3 | `fig03_delta_lst_basemap.png` | Borough ΔLST choropleth with basemap |
| Fig 4 | `fig04_ndvi_comparison.png` | NDVI maps, 2015 & 2023 |
| Fig 5 | `fig05_hotspot_boroughs.png` | Top 10 boroughs by ΔLST bar chart |
| Fig 6 | `fig06_lst_ndvi_scatter.png` | Pixel-level LST vs NDVI scatter (hexbin) |
| Fig 7 | `fig07_borough_delta_scatter.png` | Borough ΔLST vs ΔNDVI scatter |

All files saved to `outputs/figures/` at 300 DPI.